# π0.5 LIBERO Replication — Colab Notebook

Full pipeline: baseline evaluation → LoRA fine-tuning → periodic checkpoint evaluation.

**Before starting:**
- Set runtime to GPU: Runtime → Change runtime type → T4 GPU (free) or A100 (pay-per-use)
- Have your WandB API key from [wandb.ai/authorize](https://wandb.ai/authorize)

**Drive storage used:**
| What | Size |
|---|---|
| Latest checkpoint (only one kept at a time) | ~4.8 GB |
| Dataset cache | ~315 MB |
| Eval results + videos | ~50 MB |

**Section guide:**
- **§0 Config** — fill in once, never re-run
- **§1 Session Setup** — run at the start of every Colab session
- **§2 One-time Init** — run only on the very first session
- **§3 Baseline Eval** — run once before any fine-tuning
- **§4 Training** — run to train; re-run with resume after disconnect
- **§5 Checkpoint Eval** — run after each training block
- **§6 Results** — compare all evaluated checkpoints

---
## §0 · Config — fill in once

In [ ]:
# ── Paste your tokens here ────────────────────────────────────────────────────
WANDB_API_KEY = ""          # from wandb.ai/authorize
HF_TOKEN      = ""          # from huggingface.co/settings/tokens (read access is enough)

# ── Repo ──────────────────────────────────────────────────────────────────────
GITHUB_REPO = "https://github.com/LavetteSinsora/pi05-libero-replication"
HF_DATASET  = "pi05-libero/libero_object_summed_subsampling"

# ── Training knob ─────────────────────────────────────────────────────────────
# Incremental workflow: train to TRAIN_UNTIL, evaluate, then raise TRAIN_UNTIL
# and re-run §4 with RESUME = True.  Keep at multiples of 5000 (save_interval).
TRAIN_UNTIL = 5000          # first run: 5000; subsequent: 10000, 15000 … 30000
RESUME      = False         # set True after the first run

# ── Paths (do not change) ─────────────────────────────────────────────────────
REPO        = "/content/pi05-libero-replication"
DRIVE_ROOT  = "/content/drive/MyDrive/pi05_libero_replication"
# HF_HOME on Drive → HF download cache persists across sessions
HF_HOME_DIR = f"{DRIVE_ROOT}/hf_cache"
DATASET_DIR = "/content/lerobot/libero_object_summed_subsampling"
CKPT_BASE   = f"{DRIVE_ROOT}/checkpoints/pi05_libero"
EXP_BASE    = f"{DRIVE_ROOT}/experiments"

---
## §1 · Session Setup — run at the start of every session

In [ ]:
# 1a. Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 1b. Clone repo (fast: skips if already present)
import pathlib, subprocess

if not pathlib.Path(REPO).exists():
    subprocess.run(
        ["git", "clone", "--recurse-submodules", GITHUB_REPO, REPO],
        check=True,
    )
    print("Cloned.")
else:
    subprocess.run(["git", "-C", REPO, "pull", "--recurse-submodules"], check=True)
    print("Pulled latest.")

In [ ]:
# 1c. System libraries + Python packages
# apt-get is fast on repeat runs (packages already cached by Colab)
import subprocess
subprocess.run(["apt-get", "install", "-y", "-qq",
                "ffmpeg", "libgl1-mesa-glx", "libegl1-mesa"], check=True)

!pip install -q -e {REPO}/third_party/openpi
!pip install -q -e {REPO}/third_party/libero
print("Packages installed.")

In [ ]:
# 1d. Environment variables
import os

os.environ["HF_HOME"]                       = HF_HOME_DIR   # persists HF cache on Drive
os.environ["HF_LEROBOT_HOME"]               = "/content/lerobot"
os.environ["MUJOCO_GL"]                     = "egl"          # GPU off-screen rendering
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.9"

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

# Create Drive directories
for d in [HF_HOME_DIR, CKPT_BASE, EXP_BASE]:
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)

print("Env vars set.")

In [ ]:
# 1e. WandB login
import os
if WANDB_API_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY
    print("WandB key set from config.")
else:
    import wandb
    wandb.login()   # shows interactive widget

In [ ]:
# 1f. Dataset — restore from Drive cache (or download from HF Hub on first session)
# HF_HOME is on Drive, so snapshot_download writes to Drive and skips re-download
# on subsequent sessions.  The fast local copy to /content/lerobot is for training.
import pathlib
from huggingface_hub import snapshot_download

DATASET_LOCAL = pathlib.Path(DATASET_DIR)
DATASET_DRIVE = pathlib.Path(HF_HOME_DIR) / "lerobot" / "libero_object_summed_subsampling"

if DATASET_LOCAL.exists() and any(DATASET_LOCAL.iterdir()):
    print(f"Dataset already at {DATASET_LOCAL}")
elif DATASET_DRIVE.exists() and any(DATASET_DRIVE.iterdir()):
    print("Copying dataset from Drive cache to /content/lerobot  (fast)...")
    DATASET_LOCAL.parent.mkdir(parents=True, exist_ok=True)
    !cp -r {DATASET_DRIVE} {DATASET_LOCAL.parent}/
    print("Done.")
else:
    print("Downloading dataset from HF Hub (~315 MB)...")
    snapshot_download(
        repo_id=HF_DATASET,
        repo_type="dataset",
        local_dir=str(DATASET_LOCAL),
    )
    # Save a copy on Drive for future sessions
    DATASET_DRIVE.parent.mkdir(parents=True, exist_ok=True)
    !cp -r {DATASET_LOCAL} {DATASET_DRIVE.parent}/
    print("Dataset saved to Drive cache.")

In [ ]:
# 1g. Norm stats — restore from repo (committed) or from Drive fallback
import pathlib, shutil

NORM_STATS_REPO = pathlib.Path(
    f"{REPO}/assets/pi05_libero/pi05_libero_object_lora"
    "/libero_object_summed_subsampling/norm_stats.json"
)
NORM_STATS_DRIVE = pathlib.Path(
    f"{DRIVE_ROOT}/norm_stats/norm_stats.json"
)

if NORM_STATS_REPO.exists():
    print(f"Norm stats present in cloned repo.")
elif NORM_STATS_DRIVE.exists():
    print("Restoring norm stats from Drive...")
    NORM_STATS_REPO.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(str(NORM_STATS_DRIVE), str(NORM_STATS_REPO))
    print("Restored.")
else:
    print("WARNING: norm stats not found — run §2 (compute_norm_stats) first.")

---
## §2 · One-time Initialization — first session only

After running §2 once:
1. Norm stats are saved to Drive and restored automatically in §1g on future sessions.
2. **Optionally commit `assets/` to git** so the norm stats travel with the repo (then §1g 
   finds them in the clone automatically and the Drive fallback is never needed).

In [ ]:
# 2a. Compute norm stats (~5 min)
# Writes to: {REPO}/assets/pi05_libero/pi05_libero_object_lora/
#                    libero_object_summed_subsampling/norm_stats.json
import pathlib

NORM_STATS_REPO = pathlib.Path(
    f"{REPO}/assets/pi05_libero/pi05_libero_object_lora"
    "/libero_object_summed_subsampling/norm_stats.json"
)

if NORM_STATS_REPO.exists():
    print("Norm stats already exist — skipping.")
else:
    !cd {REPO}/third_party/openpi && \
        HF_LEROBOT_HOME=/content/lerobot \
        python scripts/compute_norm_stats.py --config-name pi05_libero_object_lora
    print("Norm stats written to", NORM_STATS_REPO)

In [ ]:
# 2b. Save norm stats to Drive (so §1g can restore them in future sessions)
import pathlib, shutil

NORM_STATS_DRIVE = pathlib.Path(f"{DRIVE_ROOT}/norm_stats/norm_stats.json")
NORM_STATS_DRIVE.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(str(NORM_STATS_REPO), str(NORM_STATS_DRIVE))
print(f"Saved to {NORM_STATS_DRIVE}")

---
## §3 · Baseline Evaluation

Evaluates the unmodified π0.5 base model on LIBERO-OBJECT using our config's transforms 
(summed subsampling, discrete state input, our norm stats).  This gives a reference 
success rate before any fine-tuning.

**Note on GCS access:** the base checkpoint is at `gs://openpi-assets/checkpoints/pi05_base`.
Run the auth cell below — Colab's default Google credentials have read access to this 
public bucket.

In [ ]:
# 3a. Authenticate with Google Cloud (needed to read from gs:// public buckets)
from google.colab import auth
auth.authenticate_user()

In [ ]:
# 3b. Run baseline benchmark
# 10 tasks × 50 trials = 500 rollouts.  Expected time: ~90–120 min on T4.
# Results saved to Drive; also logged to WandB under run name "pi05_base_benchmark".
import pathlib

BASELINE_EXP = f"{EXP_BASE}/pi05_base_benchmark"
pathlib.Path(BASELINE_EXP).mkdir(parents=True, exist_ok=True)

!cd {REPO}/third_party/openpi && \
    MUJOCO_GL=egl \
    python ../../scripts/benchmark.py \
        --config_name pi05_libero_object_lora \
        --checkpoint_dir gs://openpi-assets/checkpoints/pi05_base \
        --exp_dir {BASELINE_EXP}

In [ ]:
# 3c. Print baseline result
import json, pathlib
r = json.loads(pathlib.Path(f"{EXP_BASE}/pi05_base_benchmark/results.json").read_text())
print(f"Baseline aggregate success rate: {r['aggregate_success_rate']:.1%}")
for task, v in r["per_task"].items():
    print(f"  {task}: {v['success_rate']:.1%}")

---
## §4 · Fine-tuning

### Incremental workflow

Because `max_to_keep=1`, each new checkpoint overwrites the previous one.  
To evaluate periodically:

1. Set `TRAIN_UNTIL = 5000`, `RESUME = False` → run §4 cell
2. Run §5 (eval at step 5000)
3. Set `TRAIN_UNTIL = 10000`, `RESUME = True` → run §4 cell again
4. Run §5 (eval at step 10000)
5. Repeat until 30000 or until success rate plateaus

**After a Colab disconnect mid-training:** re-run §1, then re-run §4 with `RESUME = True` 
and `TRAIN_UNTIL` at the same value — training resumes from the last saved checkpoint.

Training progress is visible live in WandB under project `pi05_libero_replication`.

**Expected time (T4 GPU):** ~8–10 hours for the full 30k steps.

In [ ]:
# §4 · Train to TRAIN_UNTIL steps (set RESUME = True to continue from checkpoint)
# TRAIN_UNTIL and RESUME are set in §0 Config.

resume_flag = "--resume" if RESUME else ""

!cd {REPO}/third_party/openpi && \
    HF_LEROBOT_HOME=/content/lerobot \
    XLA_PYTHON_CLIENT_MEM_FRACTION=0.9 \
    python scripts/train.py pi05_libero_object_lora \
        --exp_name masked_loss_summed_subsampling \
        --checkpoint_base_dir {CKPT_BASE} \
        --num_train_steps {TRAIN_UNTIL} \
        {resume_flag}

In [ ]:
# Verify: list checkpoints saved on Drive
import pathlib
ckpt_run_dir = pathlib.Path(
    f"{CKPT_BASE}/pi05_libero_object_lora/masked_loss_summed_subsampling"
)
if ckpt_run_dir.exists():
    steps = sorted(
        int(d.name) for d in ckpt_run_dir.iterdir()
        if d.is_dir() and d.name.isdigit()
    )
    print("Checkpoints on Drive:", steps)
else:
    print("No checkpoint directory found yet.")

---
## §5 · Checkpoint Evaluation

Run this immediately after each training block, before raising `TRAIN_UNTIL` and resuming.
`EVAL_STEP` defaults to `TRAIN_UNTIL` (the checkpoint just produced).  
You can override it to re-evaluate an earlier step.

Expected time per checkpoint: ~90–120 min on T4 (500 rollouts).

In [ ]:
# §5 · Evaluate a checkpoint
import pathlib

EVAL_STEP = TRAIN_UNTIL   # override here if needed, e.g. EVAL_STEP = 10000

CKPT_DIR  = f"{CKPT_BASE}/pi05_libero_object_lora/masked_loss_summed_subsampling/{EVAL_STEP}"
EVAL_EXP  = f"{EXP_BASE}/pi05_libero_object_lora/masked_loss_summed_subsampling/step_{EVAL_STEP}"

if not pathlib.Path(CKPT_DIR).exists():
    raise FileNotFoundError(
        f"Checkpoint not found: {CKPT_DIR}\n"
        f"Available steps: {sorted(int(d.name) for d in pathlib.Path(CKPT_DIR).parent.iterdir() if d.name.isdigit())}"
    )

pathlib.Path(EVAL_EXP).mkdir(parents=True, exist_ok=True)

!cd {REPO}/third_party/openpi && \
    MUJOCO_GL=egl \
    python ../../scripts/benchmark.py \
        --config_name pi05_libero_object_lora \
        --checkpoint_dir {CKPT_DIR} \
        --exp_dir {EVAL_EXP} \
        --train_step {EVAL_STEP}

In [ ]:
# Print result for this checkpoint
import json, pathlib
r = json.loads(pathlib.Path(f"{EVAL_EXP}/results.json").read_text())
print(f"Step {EVAL_STEP} aggregate success rate: {r['aggregate_success_rate']:.1%}")
for task, v in r["per_task"].items():
    print(f"  {task}: {v['success_rate']:.1%}")

---
## §6 · Results Summary

Aggregates all evaluated checkpoints into a comparison table.  
Run at any point after one or more §5 eval cells have completed.

In [ ]:
import json, pathlib

def load_rate(results_json):
    p = pathlib.Path(results_json)
    if not p.exists():
        return None
    return json.loads(p.read_text())["aggregate_success_rate"]

rows = []

# Baseline
rate = load_rate(f"{EXP_BASE}/pi05_base_benchmark/results.json")
if rate is not None:
    rows.append(("π0.5 base (no fine-tuning)", rate))

# Fine-tuned checkpoints
lora_exp_base = pathlib.Path(
    f"{EXP_BASE}/pi05_libero_object_lora/masked_loss_summed_subsampling"
)
if lora_exp_base.exists():
    step_dirs = sorted(
        lora_exp_base.iterdir(),
        key=lambda d: int(d.name.split("_")[-1]) if d.name.startswith("step_") else -1
    )
    for step_dir in step_dirs:
        rate = load_rate(step_dir / "results.json")
        if rate is not None:
            rows.append((step_dir.name, rate))

if not rows:
    print("No results found yet. Run §3 and/or §5 first.")
else:
    print(f"{'Run':<40} {'Success Rate':>12}")
    print("-" * 53)
    baseline = None
    for name, rate in rows:
        delta = ""
        if baseline is None:
            baseline = rate
        else:
            diff = rate - baseline
            delta = f"  ({'+' if diff >= 0 else ''}{diff:.1%} vs baseline)"
        print(f"{name:<40} {rate:>11.1%}{delta}")